In [1]:
import sys; print(sys.executable)


/opt/anaconda3/bin/python


In [2]:
%pip install llama-cpp-python


Note: you may need to restart the kernel to use updated packages.


In [3]:
from llama_cpp import Llama
print("llama-cpp is installed and ready!")


llama-cpp is installed and ready!


In [4]:
llm = Llama(model_path="/Users/piyushbhattarai/Downloads/phi-3-mini-4k-instruct-Q4.gguf", n_gpu_layers=9999)
out = llm.create_chat_completion(
    messages=[{"role":"user","content":"Say Hi in one word"}],
    max_tokens=5
)
print(out["choices"][0]["message"]["content"])


llama_model_load_from_file_impl: using device Metal (Apple M1) - 5461 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 195 tensors from /Users/piyushbhattarai/Downloads/phi-3-mini-4k-instruct-Q4.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:                               general.name str              = Phi3
llama_model_loader: - kv   2:                        phi3.context_length u32              = 4096
llama_model_loader: - kv   3:                      phi3.embedding_length u32              = 3072
llama_model_loader: - kv   4:                   phi3.feed_forward_length u32              = 8192
llama_model_loader: - kv   5:                           phi3.block_count u32              = 32
llama_model_loader: - kv   6:                  phi3.attention.head_cou

 Hello


In [18]:
# %% [markdown]
# Student Decision Generator — Program-Agnostic, Metadata-Driven
# - Program details fully driven by LLM metadata from program description
# - No hard-coded Python/SQL/analytics prerequisites
# - Uses core_prerequisites + focus_areas for fit scoring
# - Inferred "rigor" from academic background is generic (not program-specific)
# - Outputs: decision, explanation, model_confidence, program_alignment, confidence_interval

# %%
import os, json, re, random
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm import tqdm
from IPython.display import display, FileLink
from llama_cpp import Llama

# ========== CONFIG — UPDATE THESE ==========
LLM_PATH = "/Users/piyushbhattarai/Downloads/phi-3-mini-4k-instruct-Q4.gguf"

CSV_PATH = "/Users/piyushbhattarai/Downloads/SPS_Files/sample_student_profiles.csv"
PROGRAM_TXT_PATH = "/Users/piyushbhattarai/Downloads/SPS_Files/product_description.txt"

# llama.cpp runtime
CTX_LEN = 4096
N_THREADS = 0              # 0 = auto
N_GPU_LAYERS = 9999        # 0 = CPU only
SEED = 42
random.seed(SEED)

# Generation
TEMPERATURE = 0.5
TOP_P = 0.9
MAX_NEW_TOKENS = 200

# Required CSV columns; 'domain' optional
REQUIRED_COLUMNS = [
    "academic_background",
    "academic_interests",
    "professional_interests",
    "previous_work_experience",
]
OPTIONAL_DOMAIN_COL = "domain"

# Style
MAX_WORDS = 65
MIN_SENTS, MAX_SENTS = 1, 3

# Program domains (still a generic fixed set — but logic is NOT tied to any one)
PROGRAM_DOMAINS = [
    "analytics",
    "computer_science",
    "economics",
    "finance",
    "mba",
    "business",
    "public_policy",
    "healthcare",
    "social_sciences",
    "other"
]

PRINT_SAMPLE_ROWS = 5

# ========== LOAD MODEL ==========
if not Path(LLM_PATH).exists():
    raise FileNotFoundError(f"GGUF model not found: {LLM_PATH}")

print("Loading model…")
llm = Llama(
    model_path=LLM_PATH,
    n_ctx=CTX_LEN,
    n_threads=N_THREADS,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
    seed=SEED
)
print(f"Loaded: {LLM_PATH}")

# ========== BASIC HELPERS ==========
JSON_OBJ = re.compile(r"\{.*\}", re.DOTALL)
FIRST_PERSON_RE = re.compile(r"\b(I|I'm|I’ve|I’d|my|me)\b", re.IGNORECASE)

STOPWORDS = {
    "the","and","of","in","for","to","a","an","on","with","at","by",
    "from","into","about","as","is","are","this","that","these","those",
    "my","our","your","their","through","using","use","based"
}

def tokenize(text: str):
    tokens = re.split(r"[^a-zA-Z]+", (text or "").lower())
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]

def chat_once(system_prompt: str, user_prompt: str,
              temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS) -> str:
    out = llm.create_chat_completion(
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt},
        ],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return out["choices"][0]["message"]["content"]

def trim_words(s: str, max_words=MAX_WORDS) -> str:
    w = (s or "").split()
    if not w:
        return ""
    s2 = " ".join(w[:max_words])
    if not s2.endswith((".", "!", "?")):
        s2 += "."
    return s2

# ========== DEGREE TYPE (RULE-BASED) ==========
def extract_degree_type(desc: str) -> str:
    text = (desc or "").lower()

    phd_tokens = [
        "phd", "ph.d", "doctor of philosophy", "doctoral program", "doctoral degree"
    ]
    if any(tok in text for tok in phd_tokens):
        return "PhD"

    masters_tokens = [
        "master of science", "master of arts", "master’s", "master's",
        "ms in", "m.s. in", "msc", "graduate program", "graduate degree"
    ]
    if any(tok in text for tok in masters_tokens):
        return "Masters"

    bachelor_tokens = [
        "bachelor of", "undergraduate program", "undergraduate degree",
        "b.sc", "b.s.", "ba in", "b.a. in"
    ]
    if any(tok in text for tok in bachelor_tokens):
        return "Bachelors/undergraduate"

    return "Masters"

# ========== PROGRAM METADATA EXTRACTION ==========
def parse_program_metadata_json(text: str):
    m = JSON_OBJ.search(text or "")
    cand = m.group(0) if m else (text or "")
    try:
        data = json.loads(cand)
        if not isinstance(data, dict):
            return None

        name = str(data.get("program_name", "")).strip()
        domain = str(data.get("program_domain", "")).strip().lower()
        if domain not in PROGRAM_DOMAINS:
            domain = "other"

        def norm_list(x):
            if isinstance(x, list):
                return [str(v).strip() for v in x if str(v).strip()]
            if isinstance(x, str) and x.strip():
                return [p.strip() for p in x.split(",") if p.strip()]
            return []

        return {
            "program_name": name or "this graduate program",
            "program_domain": domain,
            "focus_areas": norm_list(data.get("focus_areas", [])),
            "core_prerequisites": norm_list(data.get("core_prerequisites", [])),
            "preferred_backgrounds": norm_list(data.get("preferred_backgrounds", [])),
        }
    except Exception:
        return None

def regex_program_name(desc: str) -> str | None:
    text = desc.strip()
    m = re.search(r"(Master of [A-Za-z ]+ in [^,\.]+)", text)
    if m:
        return m.group(1).strip()
    m2 = re.search(r"(Bachelor of [A-Za-z ]+ in [^,\.]+)", text)
    if m2:
        return m2.group(1).strip()
    m3 = re.search(r"(Doctor of Philosophy in [^,\.]+)", text)
    if m3:
        return m3.group(1).strip()
    m4 = re.search(r"In ([^.]+?program)", text)
    if m4:
        return m4.group(1).strip()
    return None

def clean_focus_areas(meta: dict, program_description: str) -> list[str]:
    """
    Generic cleaning:
    - deduplicate
    - keep as-is (we're no longer enforcing analytics-specific filtering)
    - limit to 6 to keep things readable
    """
    focus = meta.get("focus_areas", []) or []
    cleaned = [f.strip() for f in focus if f.strip()]
    cleaned = list(dict.fromkeys(cleaned))
    return cleaned[:6]

def ensure_core_prereqs(meta: dict, program_description: str) -> list[str]:
    """
    If core_prerequisites is empty, ask the LLM again specifically for 3–6 core skills/knowledge areas.
    """
    core = meta.get("core_prerequisites", []) or []
    if core:
        return core

    sys_p = (
        "You are extracting key skills and knowledge areas from a graduate program description.\n"
        "Return 3–6 short prerequisite items in STRICT JSON as a string array."
    )
    usr_p = (
        "Program description:\n"
        f"{program_description}\n\n"
        "Return STRICT JSON like:\n"
        '["item1","item2","item3"]'
    )
    txt = chat_once(sys_p, usr_p, temperature=0.0, top_p=1.0, max_tokens=200)
    m = JSON_OBJ.search(txt or "")
    cand = m.group(0) if m else (txt or "")
    try:
        arr = json.loads(cand)
        if isinstance(arr, list):
            items = [str(x).strip() for x in arr if str(x).strip()]
            if items:
                return items[:6]
    except Exception:
        pass
    return core  # may be empty

def extract_program_metadata(program_description: str):
    system_p = (
        "You are analyzing a graduate program description. "
        "Extract minimal metadata in STRICT JSON with the required keys.\n"
        "Allowed program_domain values (choose exactly one): "
        + ", ".join([f'"{d}"' for d in PROGRAM_DOMAINS]) + "."
    )
    user_p = (
        "Read the following graduate program description and return metadata:\n\n"
        f"{program_description}\n\n"
        "Return STRICT JSON with this schema:\n"
        "{\n"
        '  "program_name": "string",\n'
        '  "program_domain": "analytics|computer_science|economics|finance|mba|business|public_policy|healthcare|social_sciences|other",\n'
        '  "focus_areas": ["..."],\n'
        '  "core_prerequisites": ["skills or knowledge areas students should have before starting"],\n'
        '  "preferred_backgrounds": ["degrees, majors, or experiences that are especially suitable"]\n'
        "}"
    )
    txt = chat_once(system_p, user_p, temperature=0.0, top_p=1.0, max_tokens=256)
    meta = parse_program_metadata_json(txt)

    if meta is None:
        rep_txt = chat_once(
            "You convert to strict JSON with keys {program_name, program_domain, focus_areas, core_prerequisites, preferred_backgrounds} only.",
            "Convert this to STRICT JSON with only those keys:\n\n" + (txt or ""),
            temperature=0.0, top_p=1.0, max_tokens=256
        )
        meta = parse_program_metadata_json(rep_txt)

    if meta is None:
        fallback_name = regex_program_name(program_description) or "this graduate program"
        meta = {
            "program_name": fallback_name,
            "program_domain": "other",
            "focus_areas": [],
            "core_prerequisites": [],
            "preferred_backgrounds": [],
        }

    if meta["program_name"] == "this graduate program":
        rn = regex_program_name(program_description)
        if rn:
            meta["program_name"] = rn

    meta["focus_areas"] = clean_focus_areas(meta, program_description)
    meta["core_prerequisites"] = ensure_core_prereqs(meta, program_description)
    return meta

# ========== DECISION JSON (SIMPLE) ==========
def parse_decision_json_simple(text: str):
    m = JSON_OBJ.search(text or "")
    cand = m.group(0) if m else (text or "")
    try:
        data = json.loads(cand)
        if not isinstance(data, dict):
            return None
        if "decision" not in data or "explanation" not in data:
            return None
        d = str(data["decision"]).strip().lower()
        decision = "Yes" if d.startswith("y") else ("No" if d.startswith("n") else str(data["decision"]))
        explanation = str(data["explanation"]).strip()
        return {"decision": decision, "explanation": explanation}
    except Exception:
        return None

# ========== PROMPTS FOR PER-STUDENT DECISION ==========
TONE_HINTS = [
    "sound cautiously optimistic but honest about gaps",
    "sound very practical and straightforward",
    "sound quietly confident but not overselling yourself",
    "sound a bit cautious, focusing on what you still need to learn",
    "sound reflective, connecting the program to your long-term goals",
]

OPENING_HINTS = [
    "You can start from a project, a class, or a goal that feels most relevant.",
    "You can start from a specific academic interest or course that connects to the program.",
    "You can start from your professional or internship experiences.",
    "You can start from a gap you notice in your skills and how that affects your decision.",
]

def build_program_summary(program_meta: dict, degree_type: str) -> str:
    parts = []
    name = program_meta.get("program_name","this graduate program")
    domain = program_meta.get("program_domain","other")
    core = program_meta.get("core_prerequisites",[])
    focus = program_meta.get("focus_areas",[])

    parts.append(f"Name: {name}")
    parts.append(f"Degree type: {degree_type}")
    parts.append(f"Domain: {domain}")
    if focus:
        parts.append("Key themes: " + ", ".join(focus))
    if core:
        parts.append("Key skills or knowledge expected: " + ", ".join(core))
    return " | ".join(parts)

def build_decision_system_prompt(program_meta: dict, degree_type: str) -> str:
    name = program_meta.get("program_name","this graduate program")
    domain = program_meta.get("program_domain","other")
    return (
        f"You are the internal voice of a prospective student deciding whether to apply to "
        f"\"{name}\" (degree type: {degree_type}, domain: {domain}).\n"
        "You must return STRICT JSON with keys {decision, explanation}.\n"
        "decision: 'Yes' or 'No'.\n"
        "explanation: 1–3 sentences, first-person (I/me/my), natural and human, ideally around 55 words and ≤65 words.\n"
        "Avoid copying exact wording from these instructions or the program summary; speak like a real student.\n"
        "Vary your phrasing between different students; avoid repeating the same opening (e.g., 'My background in...').\n"
        "Base your reasoning on how well the student's skills, experiences, and goals match the program’s skills and themes."
    )

def build_decision_user_prompt(program_description: str, program_meta: dict,
                               degree_type: str, profile: dict) -> str:
    summary = build_program_summary(program_meta, degree_type)
    tone = random.choice(TONE_HINTS)
    opening = random.choice(OPENING_HINTS)
    return (
        "Decide if this student would realistically want to apply and whether they seem reasonably prepared.\n"
        f"Tone: {tone}.\n"
        f"{opening}\n\n"
        "Program summary:\n"
        f"{summary}\n\n"
        "Program description (for context):\n"
        f"{program_description}\n\n"
        "Student profile (JSON):\n"
        f"{json.dumps(profile, ensure_ascii=False)}\n\n"
        "Return STRICT JSON with this structure:\n"
        "{\n"
        '  "decision": "Yes" or "No",\n'
        '  "explanation": "1–3 sentences, first-person, ≤65 words, varied phrasing"\n'
        "}"
    )

# ========== EXPLANATION POST-PROCESS ==========
BANNED_PREFIXES = [
    "strict json", "convert this", "rewrite the explanation", "original explanation",
]

def remove_prompt_artifacts(text: str) -> str:
    if not text:
        return text
    t = text.strip()
    parts = re.split(r'([.!?])', t)
    cleaned_chunks = []
    for i in range(0, len(parts), 2):
        chunk = parts[i].strip()
        if not chunk:
            continue
        lower = chunk.lower()
        if any(lower.startswith(bp) for bp in BANNED_PREFIXES):
            continue
        cleaned_chunks.append(chunk)
        if i + 1 < len(parts) and parts[i+1] in ".!?":
            cleaned_chunks.append(parts[i+1])
    cleaned = "".join(cleaned_chunks).strip()
    return cleaned or t

def ensure_first_person(s: str) -> str:
    if FIRST_PERSON_RE.search(s or ""):
        return s
    t = re.sub(r"^The student\b", "I", s or "", flags=re.IGNORECASE)
    if t != (s or ""):
        return t
    return "From my perspective, " + (s or "").lstrip()

def fix_grammar_minimal(s: str) -> str:
    if not s:
        return s
    t = s.strip()

    t = re.sub(r"\b[Ii]\s+my\b", "In my", t)
    t = re.sub(r"\bi['’`]?m\b", "I'm", t, flags=re.IGNORECASE)
    t = re.sub(r"\bi\b", "I", t)
    t = re.sub(r"\bwith a experience\b", "with experience", t, flags=re.IGNORECASE)
    t = re.sub(r"\ba experience\b", "an experience", t, flags=re.IGNORECASE)
    t = re.sub(r"\binterested for\b", "interested in", t, flags=re.IGNORECASE)

    t = t.replace("core prerequisites", "key requirements")

    t = re.sub(r"\s+([,.!?])", r"\1", t)
    t = re.sub(r"([.!?])([A-Za-z])", r"\1 \2", t)
    t = re.sub(r"\s+", " ", t)

    if not t.endswith((".", "!", "?")):
        t += "."
    return t

def limit_sentences(s: str) -> str:
    text = (s or "").strip()
    if not text:
        return text
    parts = re.split(r'([.!?])', text)
    out_parts = []
    sentence_count = 0
    for i in range(0, len(parts), 2):
        chunk = parts[i].strip()
        if not chunk:
            continue
        if sentence_count >= MAX_SENTS:
            break
        out_parts.append(chunk)
        if i + 1 < len(parts) and parts[i+1] in ".!?":
            out_parts.append(parts[i+1])
        else:
            out_parts.append(".")
        sentence_count += 1
    out = "".join(out_parts).strip()
    return trim_words(out, MAX_WORDS)

def maybe_rewrite_opening_for_variety(s: str, prob: float = 0.8) -> str:
    text = s.lstrip()
    lower = text.lower()
    candidates = [
        "my background in ",
        "with a background in ",
        "with my background in ",
        "given my background in ",
        "while my background is ",
    ]
    if any(lower.startswith(c) for c in candidates) and random.random() < prob:
        alt_openers = [
            "Studying ",
            "Working in ",
            "Spending time in ",
            "My studies in ",
            "Over the past few years, ",
            "Looking ahead, ",
            "Because I’ve spent time in ",
        ]
        alt = random.choice(alt_openers)
        match = next(c for c in candidates if lower.startswith(c))
        new = alt + text[len(match):]
        leading_ws = s[:len(s) - len(s.lstrip())]
        return leading_ws + new
    return s

def postprocess_explanation(expl: str) -> str:
    t = (expl or "").strip()
    t = remove_prompt_artifacts(t)
    t = ensure_first_person(t)
    t = fix_grammar_minimal(t)
    t = limit_sentences(t)
    t = maybe_rewrite_opening_for_variety(t)
    return t

# ========== HEURISTICS: FIT SCORES (METADATA-DRIVEN) ==========
def inferred_rigor_from_background(bg: str) -> float:
    """
    Very generic heuristic:
    - Quant/technical majors → higher rigor
    - Social sciences / business → medium
    - Others → low but non-zero if something is there
    """
    text = (bg or "").lower()
    high = [
        "mathematics","math","statistics","biostatistics",
        "computer science","cs","software","engineering",
        "physics","data science","data analytics",
        "economics","econometric","quantitative","machine learning",
        "operations research","actuarial"
    ]
    medium = [
        "business","management","psychology","biology",
        "public policy","sociology","marketing","accounting",
        "chemistry","environmental","health","political science",
        "finance"
    ]
    if any(w in text for w in high):
        return 0.8
    if any(w in text for w in medium):
        return 0.5
    if text.strip():
        return 0.2
    return 0.0

def prereq_skill_fit(profile: dict, program_meta: dict) -> float:
    """
    Skill fit based on:
    - overlap between student's text and program core_prerequisites (LLM-generated)
    - plus a generic inferred rigor from academic background.
    """
    text_all = " ".join(str(profile.get(k,"")) for k in profile.keys())
    tokens_profile = set(tokenize(text_all))
    core = [c for c in (program_meta.get("core_prerequisites",[]) or []) if c]

    hits = 0
    for c in core:
        tokens_c = set(tokenize(c))
        if tokens_c and tokens_profile.intersection(tokens_c):
            hits += 1
    base_score = hits / len(core) if core else 0.0

    rigor = inferred_rigor_from_background(profile.get("academic_background",""))
    skill_fit = 0.6*base_score + 0.4*rigor
    return max(0.0, min(1.0, skill_fit))

def interest_fit(profile: dict, program_meta: dict) -> float:
    """
    Interest fit based on:
    - overlap between academic/professional interests
    - and tokens from program_name, program_domain, focus_areas.
    All of these come from metadata or the description, not from hard-coded domain lists.
    """
    ai = (profile.get("academic_interests","") or "")
    pi = (profile.get("professional_interests","") or "")
    tokens_int = set(tokenize(ai + " " + pi))
    if not tokens_int:
        return 0.0

    kw = []
    kw += tokenize(program_meta.get("program_name",""))
    kw += tokenize(program_meta.get("program_domain",""))
    for fa in program_meta.get("focus_areas", []):
        kw += tokenize(fa)

    kw = [k for k in kw if k]
    if not kw:
        return 0.0

    kw_unique = list(dict.fromkeys(kw))
    hits = sum(1 for k in kw_unique if k in tokens_int)
    score = hits / len(kw_unique)
    return max(0.0, min(1.0, score))

def experience_score(profile: dict) -> float:
    pw = str(profile.get("previous_work_experience","") or "").strip().lower()
    if pw in {"yes","y","true","1"}:
        return 1.0
    if len(pw) >= 3:
        return 0.7
    return 0.0

def program_alignment_score(skill_fit: float, interest_fit_value: float, exp_score: float) -> float:
    raw = 0.5*skill_fit + 0.3*interest_fit_value + 0.2*exp_score
    return max(0.0, min(1.0, raw))

def model_confidence_from_alignment(decision: str, skill_fit_value: float, interest_fit_value: float) -> float:
    avg_fit = 0.6*skill_fit_value + 0.4*interest_fit_value
    if decision == "Yes":
        return 0.5 + 0.4*avg_fit
    else:
        return 0.5 + 0.4*(1.0-avg_fit)

def blend_and_interval(model_score: float, align_score: float) -> tuple[float, float]:
    blend = 0.5*model_score + 0.5*align_score
    d = abs(model_score - align_score)
    width = max(0.10, min(0.30, 0.25 - 0.10 * (1 - d)))
    return blend, width

def percent_interval_str(center: float, width: float) -> str:
    lo = max(0.0, center - width/2)
    hi = min(1.0, center + width/2)
    lo_pct = int(round(lo*100))
    hi_pct = int(round(hi*100))
    lo_pct = max(0, min(100, lo_pct))
    hi_pct = max(0, min(100, hi_pct))
    return f"{lo_pct}–{hi_pct}%"

def calibrated_decision(initial_decision: str,
                        skill_fit_value: float,
                        interest_fit_value: float,
                        alignment: float) -> str:
    """
    Program-agnostic calibration:
    - Force No only when both alignment and skill are clearly low.
    - Force Yes when alignment is high.
    - Otherwise, trust the model's initial decision.
    """
    if alignment < 0.20 and skill_fit_value < 0.30:
        return "No"
    if alignment > 0.60 and interest_fit_value > 0.35:
        return "Yes"
    return initial_decision

# ========== PER-STUDENT DECISION ==========
def generate_decision(program_description: str, program_meta: dict,
                      degree_type: str, profile: dict) -> dict:
    sys_p = build_decision_system_prompt(program_meta, degree_type)
    usr_p = build_decision_user_prompt(program_description, program_meta, degree_type, profile)

    txt = chat_once(sys_p, usr_p)
    data = parse_decision_json_simple(txt)

    if data is None:
        simple_expl = chat_once(
            "You are the student's internal voice. Output ONLY 1–3 first-person sentences about whether they would apply.",
            "Student profile:\n" + json.dumps(profile, ensure_ascii=False) +
            "\nProgram description:\n" + program_description,
            temperature=0.6,
            top_p=0.9,
            max_tokens=120
        )
        expl = postprocess_explanation(simple_expl)
        return {"decision": "No", "explanation": expl}

    return data

# ========== MAIN PIPELINE ==========
def process_profiles(csv_path: str, program_txt_path: str) -> pd.DataFrame:
    if not Path(csv_path).exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    if not Path(program_txt_path).exists():
        raise FileNotFoundError(f"Program TXT not found: {program_txt_path}")

    df = pd.read_csv(csv_path)
    program_description = Path(program_txt_path).read_text(encoding="utf-8", errors="ignore").strip()

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}\nFound columns: {list(df.columns)}")

    has_domain = OPTIONAL_DOMAIN_COL in df.columns

    print("Extracting program metadata from description…")
    program_meta = extract_program_metadata(program_description)
    degree_type = extract_degree_type(program_description)
    display(program_meta)
    print("Degree type inferred:", degree_type)

    rows = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating"):
        profile = {
            "academic_background": str(row["academic_background"]),
            "academic_interests": str(row["academic_interests"]),
            "professional_interests": str(row["professional_interests"]),
            "previous_work_experience": str(row["previous_work_experience"]),
        }
        domain_value = str(row[OPTIONAL_DOMAIN_COL]) if has_domain else None

        data = generate_decision(program_description, program_meta, degree_type, profile)
        model_decision = "Yes" if data.get("decision","Yes").strip().lower().startswith("y") else "No"
        expl_raw = data.get("explanation","")
        expl = postprocess_explanation(expl_raw)

        skill = prereq_skill_fit(profile, program_meta)
        intr = interest_fit(profile, program_meta)
        exp_score = experience_score(profile)
        align = program_alignment_score(skill, intr, exp_score)

        final_decision = calibrated_decision(model_decision, skill, intr, align)

        model_score_raw = model_confidence_from_alignment(final_decision, skill, intr)
        conf_center, width = blend_and_interval(model_score_raw, align)

        model_conf_pct = int(round(model_score_raw*100))
        program_align_pct = int(round(align*100))
        conf_interval_pct = percent_interval_str(conf_center, width)

        rows.append({
            "row_index": i,
            "decision": final_decision,
            "explanation": expl,
            "model_confidence": model_conf_pct,
            "program_alignment": program_align_pct,
            "confidence_interval": conf_interval_pct,
        })

    out_df = pd.DataFrame(rows).set_index("row_index").sort_index()
    final_df = df.join(out_df, how="left")
    return final_df

# ========== RUN ==========
try:
    output_df = process_profiles(CSV_PATH, PROGRAM_TXT_PATH)
    display(output_df.head(PRINT_SAMPLE_ROWS))
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    out_name = f"student_decisions_{ts}.csv"
    output_df.to_csv(out_name, index=False)
    print(f"\nSaved: {out_name}")
    display(FileLink(out_name))
except Exception as e:
    print(f"❌ Error: {e}")


ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h80           (not supported)
ggml

Loading model…
Loaded: /Users/piyushbhattarai/Downloads/phi-3-mini-4k-instruct-Q4.gguf
Extracting program metadata from description…


{'program_name': 'Master of Science in Analytics program',
 'program_domain': 'other',
 'focus_areas': [],
 'core_prerequisites': [],
 'preferred_backgrounds': []}

Degree type inferred: Masters


Generating: 100%|██████████| 20/20 [02:15<00:00,  6.78s/it]


,gender,age,academic_background,race,academic_interests,professional_interests,previous_work_experience,decision,explanation,model_confidence,program_alignment,confidence_interval
0,Female,22,Computer Science,Asian,Machine Learning and Data Mining,Software Development and AI Research,Yes,Yes,As a Computer Science student with a focus on ...,58,36,38–55%
1,Male,24,Economics,White,Business Analytics,Consulting,No,Yes,"Looking ahead, Economics and interest in Busin...",61,22,32–51%
2,Non-binary,21,Statistics,Black,Data Science,Research,Yes,Yes,Studying Statistics and interest in Data Scien...,61,42,43–60%
3,Female,23,Psychology,Hispanic,Behavioral Analytics,UX Research,No,No,While my background in Psychology and interest...,82,16,38–60%
4,Male,26,Engineering,Asian,Robotics,Product Development,Yes,Yes,"From my perspective, With an engineering backg...",58,36,38–55%



Saved: student_decisions_20251111-211905.csv


/Users/piyushbhattarai/student_decisions_20251111-211905.csv